In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

In [4]:
train = pd.read_csv('/Users/shubhamgupta/Documents/Projects/spaceship-titanic/data/train.csv')
test = pd.read_csv('/Users/shubhamgupta/Documents/Projects/spaceship-titanic/data/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
train.head()
train.info()

Train shape: (8693, 14)
Test shape: (4277, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [5]:
train.isnull().sum()
train['Transported'].value_counts(normalize=True)

Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64

In [ ]:
def preprocess(df, is_train=True):
    df = df.copy()
    
    df['Cabin_Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0] if pd.notna(x) else 'Unknown')
    df['Cabin_Side'] = df['Cabin'].apply(lambda x: x.split('/')[2] if pd.notna(x) else 'Unknown')
    
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')
    
    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpending'] = df[spending_cols].sum(axis=1)
    df['NoSpending'] = (df['TotalSpending'] == 0).astype(int)
    
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['CryoSleep'] = df['CryoSleep'].fillna(False).astype(int)
    df['VIP'] = df['VIP'].fillna(False).astype(int)
    for col in spending_cols:
        df[col] = df[col].fillna(0)
    df['HomePlanet'] = df['HomePlanet'].fillna('Unknown')
    df['Destination'] = df['Destination'].fillna('Unknown')
    
    for col in ['HomePlanet', 'Destination', 'Cabin_Deck', 'Cabin_Side']:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    
    feature_cols = [
        'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
        'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
        'Cabin_Deck', 'Cabin_Side', 'GroupSize', 'TotalSpending', 'NoSpending'
    ]
    
    X = df[feature_cols]
    
    if is_train:
        y = df['Transported'].astype(int)
        return X, y
    else:
        return X

In [7]:
X, y = preprocess(train, is_train=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Training set: {X_train_s.shape}")
print(f"Test set: {X_test_s.shape}")

Training set: (6954, 15)
Test set: (1739, 15)


In [8]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Accuracy: {acc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Not Transported', 'Transported']))

Logistic Regression Accuracy: 0.7895

                 precision    recall  f1-score   support

Not Transported       0.79      0.78      0.79       863
    Transported       0.79      0.80      0.79       876

       accuracy                           0.79      1739
      macro avg       0.79      0.79      0.79      1739
   weighted avg       0.79      0.79      0.79      1739



In [ ]:
import os

X_submission = preprocess(test, is_train=False)
X_submission_s = scaler.transform(X_submission)

preds = model.predict(X_submission_s)

passenger_ids = test['PassengerId']
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': preds.astype(bool)
})

os.makedirs('results', exist_ok=True)
submission.to_csv('results/01_logreg_submission.csv', index=False)

print('Submission saved: results/01_logreg_submission.csv')
submission.head()

Submission saved: results/01_logreg_submission.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
